# 03 — Forecast Evaluation

Compare the Heston Monte Carlo forecast of next-horizon realized variance against the variance actually realized over the same window. Pulls calibration history from `results/calibrations.csv` if the backtest has been run.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.monte_carlo import forecast_realized_variance
from src.calibration import calibrate_heston
from src.config import load_config, resolve_project_paths
from src.data_loader import load_all_market_data

config = load_config(ROOT / 'config.json')
paths = resolve_project_paths(config, project_root=ROOT)
bundle = load_all_market_data(paths)
as_of = pd.Timestamp('2019-01-02')
params = calibrate_heston(bundle.underlying, as_of, lookback_days=504)
spot = float(bundle.underlying.loc[bundle.underlying['date'] <= as_of, 'close'].iloc[-1])
forecast = forecast_realized_variance(spot, params, horizon_days=10, n_paths=2000, as_of_date=as_of, random_seed=42)
forecast